# Doyle Exercise 4.15: synthetic hybrid calibration and validation

This companion to `workbook.ipynb` uses the same native reactor and the existing shared estimator. The forward workbook retains its separate simulation workflow. A separately configured hybrid has `mu = mu_Monod * exp(theta*S/(S+1 g/L))`; biomass and substrate use the same corrected growth. Only dimensionless theta is fitted. The mechanistic parameters, initial conditions and feeds are treated as exact.

The existing generator independently integrates concentration balances with DOP853 and Radau. The declared design uses whole training experiments at D=0.06/h and 0.14/h and a held-out experiment at D=0.12/h. One seeded 1% noise realization is selected, with the declared 0.001 g/L absolute component. It is not a biological experiment or a statistical coverage study.

This workbook saves a native `CalibrationRecord` with training-only experiments and provenance, alongside native model JSON, CSV/manifest assets and a numerical verification report. Replay uses the shared record APIs; held-out observations never enter the fit.


### Neutral kinetic correction

The kinetic input files expose `parameters.corrections.theta = 0` (dimensionless).
The correction multiplies the selected substrate-dependent rate by
`exp(theta * substrate / (substrate + reference_concentration))`; zero preserves
the original rate. Each declaration explicitly binds the concentration and its
units. The ordinary kinetic examples reuse that rate's half-saturation
concentration as the reference scale, a configurable assumption rather than a
calibrated value. Doyle's `hybrid.json` retains its separate documented scale.
A shared theta changes all declared corrections in that file; derived yields
and balance expressions continue to use the corrected rates. Change individual
bindings to use independent coefficients if required. Selecting another kinetic
relationship also requires reviewing its input slots and correction bindings.

Pathway examples now connect empirical uptake capacities through
`model.pathways.rule_graph` and the existing `rule-graph` provider, with
`parameters.corrections.theta = 0`. Edit the capacity expression's `relationship`
and inputs to select a kinetic form; constant capacities use a parameter node.
The correction scale is the Monod affinity, or an explicit configurable
`parameters.corrections.reference_concentration = 1` [mmol/L] for constant
capacities. These are normalization assumptions, not fitted values. Monod's
`greater-select.threshold` preserves the native depletion cutoff; update it
alongside `model.pathways.depletion_tolerance`. Negative trial concentrations
use the existing zero extension. The oxygen `transfer-cap`, inventory limits,
and metabolic balances remain independent constraints, even for nonzero theta.


## Kinetics selection

Edit the named form in the input JSON, then rerun from Step 2. The processing
layer resolves the name through the existing registry; parameters, units,
stoichiometry and material destinations remain explicit. `kinetic_forms` in
Step 2 lists the available equations, aliases and required inputs. See the
example README for the exact rule identifiers and parameter paths.

Select `relationship` within `model.kinetics.rule_graph`, such as `monod`
or `haldane`. Haldane additionally requires an explicit `inhibition_constant`
input and parameter. Retain conversion, yield, inhibition and hybrid correction
nodes surrounding the selected form. Changing a form changes the scientific
model and requires new calibration/validation; it does not preserve a textbook
or experimental reproduction automatically.

The supplied calibration fits only `corrections.theta` using `inputs/hybrid.json`.
Its independently generated synthetic data assume Monod kinetics. Inspect the
selection here; changing that truth/model is a separate experiment and requires
corresponding changes to the generator and validation assumptions.


In [ ]:
# STEP 1: Select the existing example and a new output directory
from pathlib import Path
import csv
import json
import subprocess
import sys
import matplotlib.pyplot as plt

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents)
                  if (path / 'PharmaPy').is_dir()), None)
if repo_root is None:
    raise ValueError('Launch from the checkout or this example folder')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from examples.bioreactors.Doyle_Exercise_4_15 import calibrate
from PharmaPy.Bioreactors import relationship_catalog, build_bioreactor

example_dir = repo_root / 'examples/bioreactors/Doyle_Exercise_4_15'
OUTPUT_DIR = example_dir / 'outputs/calibration'
# A nonempty output directory is rejected; choose a new folder to preserve evidence.


In [ ]:
# STEP 2: Inspect explicit truth, data split and fitting assumptions
settings = json.loads((example_dir / 'inputs/calibration.json').read_text())
design = json.loads((example_dir / 'inputs/synthetic_design.json').read_text())
print(json.dumps(settings, indent=2))
print(json.dumps(design['experiments'], indent=2))
# Change noise_label/realization to select an alternative dataset, not extra replicates.
# The zero-noise control uses numerical residual scales, never division by zero sigma.

kinetic_forms = relationship_catalog()
print("Monod inputs:", kinetic_forms["saturating-uptake"]["required_inputs"])
hybrid_input = json.loads((example_dir / "inputs/hybrid.json").read_text())
print("Hybrid growth selection:", next(rule["expression"] for rule in hybrid_input["model"]["kinetics"]["rule_graph"] if rule["identifier"] == "mu_mechanistic"))


In [ ]:
# STEP 3: Generate data, fit training experiments, and evaluate untouched holdout
report = calibrate.run_workflow(OUTPUT_DIR, settings=settings, design=design)
print('Converged:', report['converged'], report['stop_criterion'])
print('Fitted parameters:', report['fitted_parameters'])
print('Training experiments:', report['fitted_experiments'])
print('Held out:', report['heldout_experiments'])
print(json.dumps(report['metrics'], indent=2))
if not report['converged']:
    raise RuntimeError('The accepted iterate is saved, but the fit did not converge')


In [ ]:
# STEP 4: Compare observations with fitted and neutral predictions
with (OUTPUT_DIR / 'predictions.csv').open() as stream:
    rows = list(csv.DictReader(stream))
figure, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
for column, experiment in enumerate(design['experiments']):
    for row_index, field in enumerate(report['field_order']):
        selected = [row for row in rows if row['experiment'] == experiment['name']
                    and row['field'] == field]
        time_h = [float(row['time_s']) / 3600. for row in selected]  # [h], exact s/h conversion
        axis = axes[row_index, column]
        axis.errorbar(time_h, [float(row['observed_g_l']) for row in selected],
                      yerr=(None if settings['noise_label'] == 'noise_0pct' else
                            [float(row['residual_scale_g_l']) for row in selected]), fmt='o', label='synthetic observations')
        axis.plot(time_h, [float(row['predicted_g_l']) for row in selected], label='fitted hybrid')
        axis.plot(time_h, [float(row['neutral_g_l']) for row in selected], '--', label='neutral theta=0')
        axis.set(title=experiment['split'] + ': ' + experiment['name'],
                 xlabel='Time [h]', ylabel=field + ' [g/L]')
axes[0, 0].legend()
figure.savefig(OUTPUT_DIR / 'calibration.png')
plt.show()


In [ ]:
# STEP 5: Reload and refit in a genuinely fresh Python process
completed = subprocess.run(
    [sys.executable, '-m', 'examples.bioreactors.Doyle_Exercise_4_15.calibrate',
     '--output', str(OUTPUT_DIR.resolve()), '--replay', '--refit'],
    cwd=repo_root, text=True, capture_output=True, check=True)
print(completed.stdout)
# Input/observation/source hashes are checked before replay; no optimizer is pickled.
# Replay reconstructs Experiment/Measurement and uses the same public callback.


In [ ]:
# STEP 6: Reuse a fitted native input through the ordinary forward API
heldout_name = report['heldout_experiments'][0]
fitted_input = json.loads((OUTPUT_DIR / 'inputs' / (heldout_name + '-fitted.json')).read_text())
assembly = build_bioreactor(fitted_input, thermo_path=OUTPUT_DIR / 'inputs/thermo.json')
histories = assembly.solve(verbose=False)
print('Reused native unit:', type(assembly.unit).__name__)
print('Fitted theta:', fitted_input['parameters']['corrections']['theta'])
print('Saved evidence:', OUTPUT_DIR)
plt.close(figure)


## Interpretation and limits

The report records actual termination, source hashes/revision, observation and parameter order, known synthetic standard deviations, integrator settings, the explicit finite-difference step, optimizer options and unscaled linearized covariance. Gaussian errors are not clipped. A zero-noise numerical scale is not an assay uncertainty. Improvement over the neutral model is reported from the observations; it is not guaranteed for every noise realization.

LM does not enforce parameter bounds: the existing provider rejects unsupported trials. `tol_fun=0` deliberately disables the residual-norm stopping rule for this noisy-data qualification; gradient, step and evaluation-budget termination remain active. The synthetic feature/domain choices and fixed mechanistic parameters are not biological calibration evidence. Analytic parameter sensitivities, empirical validation and joint identifiability are outside this example's claims.
